# 消融实验 - t+12
两个变体：
- **No-Gate**：去掉physics gate，保留extreme loss
- **No-Extreme**：保留physics gate，去掉extreme loss（lambda=0）

文件路径：`/root/autodl-tmp/`

In [1]:
# ============================================================
# Cell 1: 导入 & 共用设置
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import joblib
from sklearn.metrics import r2_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用设备:', DEVICE)

N_TIMES   = 17521  # t+12: 17532 - 12 + 1
N_GRID_UK = 2793
N_GRID_IE = 525

uk_times = pd.date_range('2020-01-01 13:00', periods=N_TIMES, freq='h')
uk_train_mask = uk_times < '2021-01-01'
uk_val_mask   = (uk_times >= '2021-01-01') & (uk_times < '2021-07-01')
uk_test_mask  = uk_times >= '2021-07-01'

ie_times = pd.date_range('2020-01-01 13:00', periods=N_TIMES, freq='h')
ie_test_mask = ie_times >= '2021-07-01'

train_idx   = np.where(np.tile(uk_train_mask, N_GRID_UK))[0]
val_idx     = np.where(np.tile(uk_val_mask,   N_GRID_UK))[0]
test_idx_uk = np.where(np.tile(uk_test_mask,  N_GRID_UK))[0]
test_idx_ie = np.where(np.tile(ie_test_mask,  N_GRID_IE))[0]

print(f'UK  Train: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx_uk):,}')
print(f'IE  Test:  {len(test_idx_ie):,}')
print('基础设置完成')

使用设备: cuda
UK  Train: 24,497,403 | Val: 12,132,792 | Test: 12,305,958
IE  Test:  2,313,150
基础设置完成


In [2]:
# ============================================================
# Cell 2: 共用 Dataset、模型、评估函数
# ============================================================
class RainfallDataset(Dataset):
    def __init__(self, X, y, idx, scaler_X, scaler_y):
        X_data = X[idx].reshape(-1, 7)
        X_data = scaler_X.transform(X_data).reshape(-1, 12, 7)
        self.X = torch.tensor(X_data, dtype=torch.float32)
        y_data = y[idx].reshape(-1, 1)
        y_data = scaler_y.transform(y_data).reshape(-1)
        self.y = torch.tensor(y_data, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

def pinball_loss(pred, target, quantile):
    err = target - pred
    return torch.where(err >= 0, quantile * err, (quantile - 1) * err).mean()

# No-Gate 模型
class NoGateQuantileLSTM(nn.Module):
    def __init__(self, input_size=7, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.2)
        self.fc_q10 = nn.Linear(hidden_size, 1)
        self.fc_q50 = nn.Linear(hidden_size, 1)
        self.fc_q90 = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        h = out[:, -1, :]
        return self.fc_q10(h).squeeze(-1), self.fc_q50(h).squeeze(-1), self.fc_q90(h).squeeze(-1)

def nogate_loss(q10, q50, q90, target, extreme_threshold=0.95):
    base_loss = (pinball_loss(q10, target, 0.1) +
                 pinball_loss(q50, target, 0.5) +
                 pinball_loss(q90, target, 0.9))
    threshold      = torch.quantile(target, extreme_threshold)
    extreme_mask   = (target > threshold).float()
    extreme_weight = 1.0 + 2.0 * extreme_mask
    weighted_loss  = (pinball_loss(q10, target, 0.1) * extreme_weight +
                      pinball_loss(q50, target, 0.5) * extreme_weight +
                      pinball_loss(q90, target, 0.9) * extreme_weight).mean()
    return base_loss + 0.5 * weighted_loss

# No-Extreme 模型
class NoExtremeQuantileLSTM(nn.Module):
    def __init__(self, input_size=7, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.2)
        self.gate_fc = nn.Sequential(
            nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
        self.fc_q10 = nn.Linear(hidden_size, 1)
        self.fc_q50 = nn.Linear(hidden_size, 1)
        self.fc_q90 = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        h    = out[:, -1, :]
        gate = self.gate_fc(x[:, -1, 4:6])
        return ((self.fc_q10(h) * gate).squeeze(-1),
                (self.fc_q50(h) * gate).squeeze(-1),
                (self.fc_q90(h) * gate).squeeze(-1),
                gate.squeeze(-1))

def noextreme_loss(q10, q50, q90, target):
    return (pinball_loss(q10, target, 0.1) +
            pinball_loss(q50, target, 0.5) +
            pinball_loss(q90, target, 0.9))

# 评估函数
def evaluate(model, loader, scaler_y, device, model_type='pg'):
    model.eval()
    all_q10, all_q50, all_q90, all_true = [], [], [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            if model_type == 'pg':
                q10, q50, q90, _ = model(X_batch)
            else:
                q10, q50, q90 = model(X_batch)
            all_q10.append(q10.cpu().numpy())
            all_q50.append(q50.cpu().numpy())
            all_q90.append(q90.cpu().numpy())
            all_true.append(y_batch.numpy())
    q10s  = np.concatenate(all_q10)
    q50s  = np.concatenate(all_q50)
    q90s  = np.concatenate(all_q90)
    trues = np.concatenate(all_true)
    q10_orig   = scaler_y.inverse_transform(q10s.reshape(-1,1)).reshape(-1)
    q50_orig   = scaler_y.inverse_transform(q50s.reshape(-1,1)).reshape(-1)
    q90_orig   = scaler_y.inverse_transform(q90s.reshape(-1,1)).reshape(-1)
    trues_orig = scaler_y.inverse_transform(trues.reshape(-1,1)).reshape(-1)
    rmse     = np.sqrt(np.mean((q50_orig - trues_orig)**2))
    r2       = r2_score(trues_orig, q50_orig)
    coverage = np.mean((trues_orig >= q10_orig) & (trues_orig <= q90_orig))
    threshold  = np.percentile(trues_orig, 95)
    obs_event  = trues_orig > threshold
    pred_event = q90_orig   > threshold
    H = np.sum( obs_event &  pred_event)
    M = np.sum( obs_event & ~pred_event)
    F = np.sum(~obs_event &  pred_event)
    csi = H / (H + M + F) if (H + M + F) > 0 else 0.0
    far = F / (H + F)     if (H + F)     > 0 else 0.0
    return rmse, r2, coverage, csi, far

def train_model(model, optimizer, loss_fn, model_type,
                train_loader, val_loader, save_path,
                epochs=30, patience=5):
    best_val_loss    = float('inf')
    patience_counter = 0
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            if model_type == 'pg':
                q10, q50, q90, _ = model(X_batch)
                loss = loss_fn(q10, q50, q90, y_batch)
            else:
                q10, q50, q90 = model(X_batch)
                loss = loss_fn(q10, q50, q90, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                if model_type == 'pg':
                    q10, q50, q90, _ = model(X_batch)
                    loss = loss_fn(q10, q50, q90, y_batch)
                else:
                    q10, q50, q90 = model(X_batch)
                    loss = loss_fn(q10, q50, q90, y_batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        print(f'Epoch {epoch+1}/{epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')
        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print(f'  ✓ 保存最优模型 (val_loss={val_loss:.4f})')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'早停！连续{patience}个epoch没有改善')
                break
    print(f'训练完成！最优Val Loss: {best_val_loss:.4f}')

def print_results(name, rmse, r2, coverage, csi, far):
    print(f'=== {name} ===')
    print(f'  RMSE:     {rmse:.6f}')
    print(f'  R²:       {r2:.4f}')
    print(f'  Coverage: {coverage:.4f}  (理想值=0.80)')
    print(f'  CSI:      {csi:.4f}')
    print(f'  FAR:      {far:.4f}')
    print()

print('所有函数和模型定义完成')

所有函数和模型定义完成


---
# Part 2：t+12 消融实验

In [6]:
import joblib
scaler_X = joblib.load('/root/scaler_X.pkl')
scaler_y = joblib.load('/root/scaler_y.pkl')

In [7]:
# ============================================================
# Cell 8: 加载 t+12 数据
# ============================================================
X_uk = np.load('/root/autodl-tmp/X_uk_lead12.npy', mmap_mode='r')
y_uk = np.load('/root/autodl-tmp/y_uk_lead12.npy', mmap_mode='r')
X_ie = np.load('/root/autodl-tmp/X_ie_lead12.npy', mmap_mode='r')
y_ie = np.load('/root/autodl-tmp/y_ie_lead12.npy', mmap_mode='r')

print('创建 t+12 DataLoader...')
train_dataset_12 = RainfallDataset(X_uk, y_uk, train_idx,   scaler_X, scaler_y)
val_dataset_12   = RainfallDataset(X_uk, y_uk, val_idx,     scaler_X, scaler_y)
test_dataset_12  = RainfallDataset(X_uk, y_uk, test_idx_uk, scaler_X, scaler_y)
ie_test_12       = RainfallDataset(X_ie, y_ie, test_idx_ie, scaler_X, scaler_y)

train_loader_12   = DataLoader(train_dataset_12, batch_size=4096, shuffle=True,  num_workers=4)
val_loader_12     = DataLoader(val_dataset_12,   batch_size=4096, shuffle=False, num_workers=4)
test_loader_12    = DataLoader(test_dataset_12,  batch_size=4096, shuffle=False, num_workers=4)
ie_test_loader_12 = DataLoader(ie_test_12,       batch_size=4096, shuffle=False, num_workers=4)
print('t+12 DataLoader 创建完成')

创建 t+12 DataLoader...
t+12 DataLoader 创建完成


In [8]:
# ============================================================
# Cell 9: t+12 No-Gate 训练
# ============================================================
print('=== t+12 No-Gate 训练开始 ===')
model_nogate_12 = NoGateQuantileLSTM().to(DEVICE)
optimizer       = torch.optim.Adam(model_nogate_12.parameters(), lr=3e-4)
train_model(model_nogate_12, optimizer, nogate_loss, 'nogate',
            train_loader_12, val_loader_12,
            save_path='/root/autodl-tmp/best_ablation_nogate_lead12.pth')

=== t+12 No-Gate 训练开始 ===
Epoch 1/30 | Train: 0.6396 | Val: 0.5468
  ✓ 保存最优模型 (val_loss=0.5468)
Epoch 2/30 | Train: 0.6003 | Val: 0.5612
Epoch 3/30 | Train: 0.5765 | Val: 0.5681
Epoch 4/30 | Train: 0.5599 | Val: 0.5748
Epoch 5/30 | Train: 0.5467 | Val: 0.5809
Epoch 6/30 | Train: 0.5362 | Val: 0.5843
早停！连续5个epoch没有改善
训练完成！最优Val Loss: 0.5468


In [9]:
# ============================================================
# Cell 10: t+12 No-Gate 评估
# ============================================================
model_nogate_12.load_state_dict(torch.load('/root/autodl-tmp/best_ablation_nogate_lead12.pth'))

r = evaluate(model_nogate_12, test_loader_12,    scaler_y, DEVICE, 'nogate')
print_results('No-Gate | UK Test (t+12)', *r)

r = evaluate(model_nogate_12, ie_test_loader_12, scaler_y, DEVICE, 'nogate')
print_results('No-Gate | IE Zero-shot (t+12)', *r)

=== No-Gate | UK Test (t+12) ===
  RMSE:     0.000370
  R²:       -0.0393
  Coverage: 0.8310  (理想值=0.80)
  CSI:      0.1063
  FAR:      0.8704

=== No-Gate | IE Zero-shot (t+12) ===
  RMSE:     0.000391
  R²:       -0.0578
  Coverage: 0.8360  (理想值=0.80)
  CSI:      0.0908
  FAR:      0.8838



In [10]:
# ============================================================
# 汇总 t+12 结果
# ============================================================
print("重新加载模型并汇总t+12结果...
")
model_nogate_12.load_state_dict(torch.load("/root/autodl-tmp/best_ablation_nogate_lead12.pth"))
model_noext_12.load_state_dict(torch.load("/root/autodl-tmp/best_ablation_noextreme_lead12.pth"))

results = {}
results["No-Gate    | UK t+12"] = evaluate(model_nogate_12, test_loader_12,    scaler_y, DEVICE, "nogate")
results["No-Gate    | IE t+12"] = evaluate(model_nogate_12, ie_test_loader_12, scaler_y, DEVICE, "nogate")
results["No-Extreme | UK t+12"] = evaluate(model_noext_12,  test_loader_12,    scaler_y, DEVICE, "pg")
results["No-Extreme | IE t+12"] = evaluate(model_noext_12,  ie_test_loader_12, scaler_y, DEVICE, "pg")

print(f"{"Model":<25} {"RMSE":>10} {"R²":>8} {"Coverage":>10} {"CSI":>8} {"FAR":>8}")
print("-" * 75)
for name, (rmse, r2, cov, csi, far) in results.items():
    print(f"{name:<25} {rmse:>10.6f} {r2:>8.4f} {cov:>10.4f} {csi:>8.4f} {far:>8.4f}")
print("
完成！")

SyntaxError: unterminated string literal (detected at line 4) (3006355909.py, line 4)